# 03 — Results

**Purpose:** the charts behind the writeup.

**Inputs:** `output/polymarket_breaks_dataset.csv` (from notebook 02).

**Outputs:** chart PNGs in `output/`.

**Prerequisites:** run `02-analysis` first.

Charts are saved at >=150 DPI with titles, axis labels, and legends. Re-running regenerates them identically.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src.transform import build_dataset
from src import analysis as A
from src.utils import OUTPUT_DIR
dataset = build_dataset(write=False)

### Figure: bracket concentration
Share of broken markets versus share of wrong-side exposure.

In [ ]:
conc = A.bracket_concentration(dataset)
bp = {r['pattern']: r for r in A.by_bracket_pattern(dataset)}
labels = ['Crypto price brackets', 'Musk tweet brackets', 'All other']
keys = ['crypto_price_bracket', 'musk_tweet_bracket', 'other']
share_n = [bp.get(k, {'breaks_pct':0})['breaks_pct'] for k in keys]
share_e = [bp.get(k, {'exposure_pct':0})['exposure_pct'] for k in keys]
colors = ['#2f6bff', '#9aa3b2', '#dde3ec']
fig, ax = plt.subplots(figsize=(9, 3.6))
for yi, shares in enumerate([share_n, share_e]):
    left = 0
    for v, c in zip(shares, colors):
        ax.barh(1-yi, v, left=left, color=c, edgecolor='white')
        if v > 0.04: ax.text(left+v/2, 1-yi, f'{v*100:.0f}%', ha='center', va='center', fontweight='bold')
        left += v
ax.set_yticks([1, 0]); ax.set_yticklabels(['Share of broken markets', 'Share of exposure'])
ax.set_xlim(0, 1); ax.set_title('Bracket compression carries half the cost')
ax.legend(labels, loc='lower center', bbox_to_anchor=(0.5, -0.35), ncol=3, frameon=False)
fig.savefig(OUTPUT_DIR / '01-bracket-concentration.png', dpi=200, bbox_inches='tight')
print('saved output/01-bracket-concentration.png')

### Figure: category divergence
Share of breaks versus share of exposure, by category.

In [ ]:
cats = A.by_category(dataset)
N = sum(c['breaks'] for c in cats); E = sum(c['exposure_usd'] for c in cats)
names = [c['category'] for c in cats]
import numpy as np
y = np.arange(len(cats))[::-1]
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(y+0.2, [-c['breaks']/N for c in cats], height=0.38, color='#9aa3b2', label='Share of breaks')
ax.barh(y-0.2, [c['exposure_usd']/E for c in cats], height=0.38, color='#2f6bff', label='Share of exposure')
ax.set_yticks(y); ax.set_yticklabels(names); ax.axvline(0, color='black', lw=0.8)
ax.set_xticks([-.4,-.2,0,.2,.4]); ax.set_xticklabels(['40%','20%','0%','20%','40%'])
ax.set_title('Frequency and cost rank categories differently'); ax.legend(frameon=False)
fig.savefig(OUTPUT_DIR / '02-category-divergence.png', dpi=200, bbox_inches='tight')
print('saved output/02-category-divergence.png')

### Figure: threshold sensitivity

In [ ]:
import csv
from src.utils import SNAPSHOTS
snaps = list(csv.DictReader(open(SNAPSHOTS, encoding='utf-8')))
ts = A.threshold_sensitivity(snaps)
fig, ax = plt.subplots(figsize=(8, 4))
xs = [f">{int(t['threshold']*100)}%" for t in ts]
ax.bar(xs, [t['breaks'] for t in ts], color='#2f6bff')
for i, t in enumerate(ts): ax.text(i, t['breaks']+10, f"{t['breaks']:,}", ha='center', fontweight='bold')
ax.set_ylabel('Broken markets'); ax.set_title('Sensitivity to the confidence threshold')
fig.savefig(OUTPUT_DIR / '03-threshold-sensitivity.png', dpi=200, bbox_inches='tight')
print('saved output/03-threshold-sensitivity.png')